# 01 — Exploratory Data Analysis: Zomato Bangalore Restaurants

This notebook explores the Zomato Bangalore Restaurants dataset.
All core logic lives in `src/dabba/` — this notebook calls those functions
and focuses on exploration and narrative.

**Dataset:** Zomato Bangalore Restaurants (Kaggle)
**Goal:** Understand data quality, distributions, and key patterns before modeling.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dabba.config import get_config
from dabba.data.loaders import load_zomato, describe_dataset
from dabba.data.cleaning import clean_zomato

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
config = get_config()

## 1. Load Raw Data

In [ ]:
df_raw = load_zomato(config)
describe_dataset(df_raw, 'Zomato (raw)')

## 2. Data Quality Check

In [ ]:
print('Shape:', df_raw.shape)
print('\nColumn types:')
print(df_raw.dtypes)
print('\nNull counts:')
print(df_raw.isnull().sum())
print('\nDuplicate rows:', df_raw.duplicated().sum())

## 3. Rating Distribution

In [ ]:
from dabba.data.cleaning import clean_zomato_rating

if 'rate' in df_raw.columns:
    cleaned_rating = clean_zomato_rating(df_raw['rate'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    df_raw['rate'].value_counts().head(10).plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Raw Rating Distribution')
    axes[0].set_xlabel('Rating')
    
    cleaned_rating.hist(bins=20, ax=axes[1], color='coral', edgecolor='black')
    axes[1].set_title('Cleaned Rating Distribution')
    axes[1].set_xlabel('Rating (float)')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Cleaned ratings: {cleaned_rating.notna().sum()} valid, {cleaned_rating.isna().sum()} null')

## 4. Cost Distribution

In [ ]:
from dabba.data.cleaning import clean_zomato_cost

cost_col = 'approx_cost(for two people)'
if cost_col in df_raw.columns:
    cleaned_cost = clean_zomato_cost(df_raw[cost_col])
    
    fig, ax = plt.subplots(figsize=(10, 5))
    cleaned_cost.hist(bins=30, ax=ax, color='green', edgecolor='black')
    ax.set_title('Cost for Two People Distribution')
    ax.set_xlabel('Cost (₹)')
    ax.set_ylabel('Count')
    plt.show()
    
    print(f'Mean: ₹{cleaned_cost.mean():.0f}, Median: ₹{cleaned_cost.median():.0f}')

## 5. Top Cuisines

In [ ]:
if 'cuisines' in df_raw.columns:
    all_cuisines = (
        df_raw['cuisines']
        .str.split(',')
        .explode()
        .str.strip()
        .value_counts()
        .head(15)
    )
    
    fig, ax = plt.subplots(figsize=(10, 6))
    all_cuisines.plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Top 15 Cuisines')
    ax.set_xlabel('Number of Restaurants')
    plt.tight_layout()
    plt.show()

## 6. Online Order & Table Booking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if 'online_order' in df_raw.columns:
    df_raw['online_order'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
    axes[0].set_title('Online Order Available')
    axes[0].set_ylabel('')

if 'book_table' in df_raw.columns:
    df_raw['book_table'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#99ff99','#ffcc99'])
    axes[1].set_title('Table Booking Available')
    axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 7. Apply Cleaning Pipeline

In [ ]:
df_clean = clean_zomato(df_raw, config)
describe_dataset(df_clean, 'Zomato (cleaned)')
print(f'\nRows after cleaning: {len(df_clean)} (from {len(df_raw)})')

## 8. Correlation Heatmap

In [ ]:
numeric_cols = df_clean.select_dtypes(include='number').columns
if len(numeric_cols) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(df_clean[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
    ax.set_title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

## Key Takeaways

- Document your findings here after running the notebook
- Note any data quality issues discovered
- Highlight distributions and patterns relevant to modeling